# Baseline models

Rule baselines, authentication-only, and complete-telemetry combined models (README section 8).

In [ ]:
import sys
from pathlib import Path

import yaml

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

CONFIG = yaml.safe_load((ROOT / "configs" / "experiment.yaml").read_text())

In [ ]:
import json

import pandas as pd
import polars as pl

from knightglass.evaluation import recall_at_daily_budget
from knightglass.models import (
    ModelRanker, RuleRanker, feature_sets, fit_cached, labeled_days, load_sample, score_period,
)

PROCESSED = ROOT / CONFIG["data"]["processed_dir"]
FEATURES = PROCESSED / "features"
SCORES = PROCESSED / "scores"
FITTED = PROCESSED / "models"
METRICS = ROOT / "results" / "metrics"
METRICS.mkdir(parents=True, exist_ok=True)

SEED = CONFIG["seed"]
BUDGETS = CONFIG["evaluation"]["daily_review_budgets"]
KEEP = max(BUDGETS)
SPLITS = CONFIG["splits"]
SETS = feature_sets(CONFIG["features"]["windows_seconds"])

# Days without labeled rows cannot change budget recall, so only these are scored.
DAYS = {period: labeled_days(FEATURES, SPLITS[period]) for period in ("validation", "test")}


def recall_table(ranked, include_possible=False):
    return {name: {b: recall_at_daily_budget(r, b, include_possible)["recall"] for b in BUDGETS}
            for name, r in ranked.items()}

## Rule baselines

Fixed heuristics with no fitting, so they are scored on validation and test directly (README section 8). Scores are cached per model and day under `data/processed/scores`, so an interrupted run resumes where it stopped.

In [ ]:
rules = {"rule_novelty": RuleRanker("novelty"), "rule_rate": RuleRanker("rate")}
rule_ranked = {period: score_period(rules, FEATURES, DAYS[period], KEEP, SEED, SCORES / period)
               for period in ("validation", "test")}
pd.DataFrame({period: {name: r[500] for name, r in recall_table(ranked).items()}
              for period, ranked in rule_ranked.items()})

## Fit the settings grid and score validation

Every feature set, model kind, setting and weighting is fit on the training sample (days 6–12) and scored on every row of the validation days with labeled rows. Fitted models are cached under `data/processed/models`. `auth_plus_collection` is the collection-artifact check from README section 6.

In [ ]:
from itertools import product

train_sample = pl.read_parquet(PROCESSED / "samples" / "train.parquet")
columns = sorted({c for numeric in SETS.values() for c in ModelRanker("logistic", numeric, {}, False, SEED).columns})
train = load_sample(FEATURES, train_sample, columns)

grid = {}
for set_name, numeric in SETS.items():
    for kind in ("logistic", "hist_gb"):
        for i, params in enumerate(CONFIG["models"][kind]):
            for weighting in CONFIG["models"]["weighting"]:
                name = f"{set_name}__{kind}__{i}__{weighting}"
                grid[name] = fit_cached(
                    name, ModelRanker(kind, numeric, params, weighting == "inclusion", SEED),
                    train, train["weight"], FITTED / "train")

validation = score_period(grid, FEATURES, DAYS["validation"], KEEP, SEED, SCORES / "validation")
validation_recall = recall_table(validation)
pd.DataFrame(validation_recall).T

## Select on validation, refit, and score test once

For each feature set and model kind, keep the setting with the highest mean validation recall across the review budgets, refit it on days 6–20, and score the test period (days 21–30) once.

The first test pass is void for the model rankers. Gradient boosting diverged (scores saturated at 0 and 1, training AUC near 0.5): without L2 regularization at this class imbalance, and with inclusion weights at the refit sample's scale (mean 258). The fix uses training-sample evidence only: `l2_regularization: 1.0` in the grid, and weights normalized to mean 1 before fitting. All model rankers were then reselected, refit and scored on test again. Rule test scores are from the first pass.


In [ ]:
import numpy as np

chosen = {}
for set_name in SETS:
    for kind in ("logistic", "hist_gb"):
        names = [n for n in grid if n.startswith(f"{set_name}__{kind}__")]
        best = max(names, key=lambda n: np.mean(list(validation_recall[n].values())))
        chosen[f"{set_name}__{kind}"] = best

refit_sample = pl.read_parquet(PROCESSED / "samples" / "refit.parquet")
refit = load_sample(FEATURES, refit_sample, columns)
final = {}
for name, best in chosen.items():
    m = grid[best]
    final[name] = fit_cached(name, ModelRanker(m.kind, m.numeric, m.params, m.weighted, SEED),
                             refit, refit["weight"], FITTED / "refit")

test = score_period(final, FEATURES, DAYS["test"], KEEP, SEED, SCORES / "test")
test = {**test, **rule_ranked["test"]}
pd.DataFrame(recall_table(test)).T

## Test recall by DNS era and with possible red-team rows

Known positives on days 27–30 fall in the DNS full-collection era (README section 6). The sensitivity check counts possible red-team rows as positives (section 4.4).

In [ ]:
cutoff = CONFIG["dns"]["full_collection_from_day"]
eras = {"before_full_dns": lambda r: r.filter(pl.col("day") < cutoff),
        "full_dns": lambda r: r.filter(pl.col("day") >= cutoff)}
by_era = {era: recall_table({n: pick(r) for n, r in test.items()}) for era, pick in eras.items()}
with_possible = recall_table(test, include_possible=True)

summary = {
    "chosen": chosen,
    "validation": {n: validation_recall[b] for n, b in chosen.items()}
                  | {n: r for n, r in recall_table(rule_ranked["validation"]).items()},
    "test": recall_table(test),
    "test_by_era": by_era,
    "test_with_possible": with_possible,
}
(METRICS / "phase3_baselines.json").write_text(json.dumps(summary, indent=2, default=str))
pd.concat({era: pd.DataFrame(t).T for era, t in by_era.items()}, axis=1)